You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [3]:
import os
from utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [10]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:

Title: The Future of Artificial Intelligence: Trends, Players, and News

Introduction:
- Brief overview of the current landscape of artificial intelligence
- Mention of the significance of staying updated on the latest trends and news in AI

Key Points:
1. Latest Trends in Artificial Intelligence
- Deep learning advancements
- AI in healthcare and medical research
- Ethical considerations in AI development

2. Key Players in the AI Industry
- Google
- Amazon
- IBM
- Microsoft

3. Noteworthy News in Artificial

I now can give a great answer

Final Answer:
# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction

Artificial Intelligence (AI) has become an integral part of our lives, revolutionizing various industries and reshaping the way we interact with technology. Staying updated on the latest trends and news in AI is crucial for tech enthusiasts, professionals in the AI industry, and students studying AI to understand the impact of this rapidly evolving technology.

## Latest Trends in Artificial Intelligence

### Deep Learning Advancements
Deep learning has been a driving force behind the recent advancements in AI, allowing machines to learn from vast amounts of data and improve their performance over time. From image recognition to natural language processing, deep learning has enabled AI systems to achieve unprecedented levels of accuracy and efficiency.

### AI in Healthcare and Medical Research
The healthcare industry has embraced AI to enhance patient care,

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction

Artificial Intelligence (AI) has become an integral part of our lives, revolutionizing various industries and reshaping the way we interact with technology. Staying updated on the latest trends and news in AI is crucial for tech enthusiasts, professionals in the AI industry, and students studying AI to understand the impact of this rapidly evolving technology.

## Latest Trends in Artificial Intelligence

### Deep Learning Advancements
Deep learning has been a driving force behind the recent advancements in AI, allowing machines to learn from vast amounts of data and improve their performance over time. From image recognition to natural language processing, deep learning has enabled AI systems to achieve unprecedented levels of accuracy and efficiency.

### AI in Healthcare and Medical Research
The healthcare industry has embraced AI to enhance patient care, optimize treatment plans, and streamline administrative processes. AI-powered technologies such as predictive analytics and robotic surgery have revolutionized healthcare delivery, leading to improved outcomes and reduced costs.

### Ethical Considerations in AI Development
As AI continues to advance, ethical considerations have become a focal point in the development and deployment of AI systems. Issues such as bias in AI algorithms, data privacy, and the impact of AI on job displacement have sparked debates on the responsible use of AI and the need for ethical guidelines in AI development.

## Key Players in the AI Industry

### Google
Google is at the forefront of AI research and innovation, leveraging its vast data resources and cutting-edge technologies to develop AI-powered products and services. From self-driving cars to personalized search recommendations, Google's AI initiatives have reshaped the way we interact with technology.

### Amazon
Amazon has integrated AI into its e-commerce platform to enhance customer experiences, optimize supply chain operations, and drive business growth. Through initiatives like Amazon Alexa and Amazon Go, the company has demonstrated the transformative potential of AI in redefining the retail industry.

### IBM
IBM is a key player in the AI industry, known for its cognitive computing platform Watson and AI-driven solutions for enterprise clients. IBM's focus on AI ethics and responsible AI deployment has positioned the company as a leader in shaping the future of AI technology.

### Microsoft
Microsoft has made significant investments in AI research and development, with a focus on democratizing AI and making it accessible to a wide range of users. From AI-powered productivity tools to cloud-based AI services, Microsoft's AI initiatives are driving innovation and empowering businesses to harness the power of AI.

## Noteworthy News in Artificial Intelligence

### Recent Breakthroughs in Natural Language Processing
Recent advancements in natural language processing have enabled AI systems to understand and generate human-like text, revolutionizing applications such as chatbots, language translation, and content creation. Breakthroughs in neural language models like GPT-3 have pushed the boundaries of AI language capabilities to new heights.

### Applications of AI in Autonomous Vehicles
AI has played a crucial role in the development of autonomous vehicles, enabling self-driving cars to navigate complex environments, make real-time decisions, and ensure passenger safety. Companies like Tesla, Waymo, and Uber are leading the charge in AI-driven transportation technologies, paving the way for a future of autonomous mobility.

### Impact of AI on Job Market Trends
The integration of AI into various industries has raised concerns about the impact of automation on job market trends. While AI has the potential to create new job opportunities and enhance productivity, it also poses challenges such as job displacement and skills gaps. Understanding the implications of AI on the workforce is essential for preparing for the future of work in an AI-driven economy.

## Conclusion

Artificial Intelligence is shaping the future of technology and transforming the way we live, work, and interact with the world around us. By staying informed on the latest trends, key players, and noteworthy news in AI, we can navigate the complexities of the AI landscape and leverage the power of AI to drive innovation and progress in our society.

## Call to Action

Stay informed on the latest AI news and developments by subscribing to our newsletter. Join our community of AI enthusiasts, professionals, and students to stay ahead of the curve and engage in meaningful discussions on the future of Artificial Intelligence.

By following this comprehensive content plan and incorporating SEO keywords naturally, we aim to provide valuable insights and information on Artificial Intelligence to our target audience, helping them navigate the rapidly evolving AI landscape and make informed decisions in the age of AI.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [13]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on YOUR TOPIC HERE.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:

Content Plan: 

Title: Exploring the Latest Trends in Sustainable Fashion

Introduction:
- Brief overview of sustainable fashion and its importance in the current fashion industry
- Introduction of key players in sustainable fashion and their impact
- Mention of noteworthy news related to sustainable fashion

Key Points:
1. Latest Trends in Sustainable Fashion:
- Sustainable materials such as organic cotton, recycled polyester, and Tencel
- Zero-waste fashion design and production techniques
- Slow fashion movement a

I now can give a great answer

Final Answer:
# Exploring the Latest Trends in Sustainable Fashion

In today's fast-paced fashion industry, the concept of sustainable fashion has gained significant traction. With increasing awareness about the environmental and social impact of traditional fashion practices, more consumers are seeking out eco-friendly and ethical alternatives. Sustainable fashion encompasses a range of practices, from using organic and recycled materials to implementing zero-waste production techniques. It not only aims to reduce the industry's carbon footprint but also promotes transparency and fair labor practices throughout the supply chain.

## Latest Trends in Sustainable Fashion

One of the key trends in sustainable fashion is the use of sustainable materials such as organic cotton, recycled polyester, and Tencel. These materials are not only environmentally friendly but also offer high-quality alternatives to traditional fabrics. Additionally, zero-waste fashion 

In [14]:
Markdown(result)

# Exploring the Latest Trends in Sustainable Fashion

In today's fast-paced fashion industry, the concept of sustainable fashion has gained significant traction. With increasing awareness about the environmental and social impact of traditional fashion practices, more consumers are seeking out eco-friendly and ethical alternatives. Sustainable fashion encompasses a range of practices, from using organic and recycled materials to implementing zero-waste production techniques. It not only aims to reduce the industry's carbon footprint but also promotes transparency and fair labor practices throughout the supply chain.

## Latest Trends in Sustainable Fashion

One of the key trends in sustainable fashion is the use of sustainable materials such as organic cotton, recycled polyester, and Tencel. These materials are not only environmentally friendly but also offer high-quality alternatives to traditional fabrics. Additionally, zero-waste fashion design and production techniques have gained popularity, minimizing the amount of textile waste generated during the manufacturing process. The slow fashion movement, which advocates for conscious consumption and the creation of capsule wardrobes, is also gaining momentum as consumers seek to reduce their environmental impact through mindful shopping habits.

## Key Players in Sustainable Fashion

Several key players in the fashion industry have made significant contributions to sustainable fashion. Stella McCartney is known for her commitment to using eco-friendly materials and implementing sustainable practices throughout her brand. Patagonia, a well-established outdoor clothing company, has been a pioneer in sustainable production practices, prioritizing environmental conservation and ethical sourcing. Eileen Fisher is another notable figure in the sustainable fashion scene, focusing on creating ethical and transparent supply chains for her clothing line.

## Noteworthy News in Sustainable Fashion

The sustainable fashion landscape is constantly evolving, with new brands and collaborations emerging to promote eco-friendly practices. Major fashion houses are also taking steps towards sustainability, launching initiatives to reduce their environmental impact and support ethical practices. The impact of sustainable fashion goes beyond the industry itself, influencing environmental conservation efforts and promoting social responsibility within the fashion community.

In conclusion, sustainable fashion is not just a passing trend but a necessary shift towards a more ethical and environmentally conscious industry. With the latest trends, key players, and noteworthy news in sustainable fashion, consumers have the opportunity to make informed choices that align with their values and contribute to a more sustainable future for the fashion industry.

Sources:
- [Stella McCartney](https://www.stellamccartney.com/)
- [Patagonia](https://www.patagonia.com/)
- [Eileen Fisher](https://www.eileenfisher.com/)
- [Business of Fashion](https://www.businessoffashion.com/)
- [Eco Warrior Princess](https://ecowarriorprincess.net/)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).